# Dataset Curation 

some of our dft calculations could have gone wrong. The way to filter this is through the E-V curves for each sample. 
- input : non curated bs
- output: curated bs regarding ev curves

TODO: bopio and bopcal needed for featurizer ! separate package for bopfox ?

## check ev-curves for goodness

In [ ]:
import sys
dataset = 'Fe-Mo'  #'Cr-Co-W'
atoms = dataset.split('-')
from Tools.DatasetTools.EVCurvesTools import *
from dependencies.bopfoxfeaturizer.BopFoxFeaturizer.Featurizer import Featurizer
import pickle
import os

In [ ]:
import re

In [ ]:
from scipy.optimize import curve_fit

In [ ]:
from ase.eos import EquationOfState, birchmurnaghan

In [ ]:
plt.rc('figure', figsize=(18, 8))
plt.rc('font', size=26)
plt.rc('xtick', labelsize=26)
plt.rc('ytick', labelsize=26)
plt.rc('axes', labelsize=30)
from matplotlib.lines import Line2D

In [ ]:
make_plots = False

In [ ]:
def get_key_for_curves(EVC: pd.core.series.Series, key='str') -> pd.core.series.Series:
    r2 = {}
    for index, data in EVC.items():
        r2[index]={}
        for params, curve in data.items():
            r2[index].update({params: curve[key]})
    return pd.Series(r2)
    

In [ ]:
PBS = pd.read_pickle(os.path.join(dataset, 'ParsedBriefsummary.pkl'))

In [ ]:
PBS.describe()

In [ ]:
PBS.B0.plot.hist(bins=100)

## Investigate ev-curves

In [ ]:
fittedcurvesloc = os.path.join(dataset, 'evcurvesfitted.json')
evcurvesloc = os.path.join(dataset,'evcurves.json' )
goodnessloc = os.path.join(dataset, 'goodness.json')
force = True#False

In [ ]:
Mo_R = 'Mo_sv53.R.NM'

In [ ]:
Fe_R = 'Fe_pv53.R.NM'

In [ ]:
if not os.path.exists(fittedcurvesloc) or force:
    if not os.path.exists(evcurvesloc) or force:
        print('redoing')

        EV = Evcurves(Indexes = PBS.index, atoms=dataset.split(), dataset = dataset)#, search_str='**/volume_relaxed/**/volume-energy.dat')
        EV.load_evcurves( deltaks = PBS['deltak'], encuts = PBS['encut'])
        EVcurves = EV.evcurves
        EVcurves.to_json(evcurvesloc)
    else:
        EVcurves = pd.read_json(evcurvesloc, typ='series')
    goodness, fiteos, r2  = get_goodness(EVcurves, r2tol = 1e-6)
    if goodness.map(lambda g: False in g.values()).all():
        goodness = invert_goodness(goodness)
    Goodness = pd.Series(goodness)
    Fits = pd.Series(fiteos)
    R2 = pd.Series(r2)
    for index, data in EVcurves.items():
        for key, evcurve in data.items():
            if index in R2.keys():
                if key in R2[index].keys():
                    data[key].update({'r2': R2[index][key], 'fit': Fits[index][key], 'IsGood': Goodness[index][key]})
    EVcurves.to_json(fittedcurvesloc)
else:
    print('B')
    EVcurves = pd.read_json(fittedcurvesloc, typ='series')
    R2 = get_key_for_curves(EVcurves, 'r2')
    Goodness = get_key_for_curves(EVcurves, 'IsGood')
    Fits = get_key_for_curves(EVcurves, 'fit')

In [ ]:
EVcurves

In [ ]:
Febcc = 'Fe_pv1.bcc.FM'

In [ ]:
figs, axs = plot_curves(EVcurves[[Febcc]], Fits[[Febcc]], R2[[Febcc]])

In [ ]:
EVcurves_df = pd.DataFrame.from_dict(EVcurves.to_dict(), orient='index')

In [ ]:
EVcurves_df.index

In [ ]:
R2_df = pd.DataFrame.from_dict(R2.to_dict(), orient='index')

In [ ]:
Fits_df = pd.DataFrame.from_dict(Fits.to_dict(), orient='index')

In [ ]:
Better_evcurves = {}

In [ ]:
for index, evcurves in EVcurves_df.iterrows():
    nonan_evcurves = evcurves.dropna()
    if len(nonan_evcurves) > 1:
        break
    nonan_evcurves[nonan_evcurves.index][0]['calc_param'] = nonan_evcurves.index[0]
    Better_evcurves[index] = nonan_evcurves[nonan_evcurves.index][0]

In [ ]:
Better_evcurves_df = pd.DataFrame.from_dict(Better_evcurves, orient='index')

In [ ]:
ev_fit_results_df = pd.DataFrame.from_dict(Better_evcurves_df.ev_fit_results.to_dict(), orient='index')

In [ ]:
fit_results = Better_evcurves_df.fit.map( lambda f : {name: val for name, val in zip(['E_murn', 'B_murn', 'Bdev_murn', 'V_murn'], f)})

In [ ]:
fit_results_df = pd.DataFrame.from_dict(fit_results.to_dict(), orient='index')

In [ ]:
if Febcc in fit_results_df.index:
    print(fit_results_df.loc[Febcc])
    print(f' Is Good : {Better_evcurves_df.loc[Febcc].IsGood}')
else:
    print(f'{Febcc} not in index')

In [ ]:
ax = fit_results_df[Better_evcurves_df.IsGood]['B_murn'].plot.hist(bins=100)
ax.set_xlabel(xlabel=r'$B_0$ (GPa)')
Ngood = Better_evcurves_df.IsGood.sum()
ax.set_title(f'{Ngood} Good Samples')

In [ ]:
indexofgoodsamples = Better_evcurves_df.index[Better_evcurves_df.IsGood]

In [ ]:
indexofbadsamples = Better_evcurves_df.index.difference(indexofgoodsamples) #[~Better_evcurves_df.IsGood]

In [ ]:
Better_evcurves_df.index.difference(PBS.index)

In [ ]:
PBS.index.difference(Better_evcurves_df.index)

In [ ]:
indexofbadsamples

In [ ]:
hist, bins = np.histogram(1-Better_evcurves_df.r2[indexofgoodsamples], bins=100)
logbins = np.logspace(np.log10(bins[0]), np.log10(bins[-1]), len(bins))
fig, ax = plt.subplots()
ax.hist(1-Better_evcurves_df.r2[indexofgoodsamples], bins=logbins)
Better_evcurves_df.r2[indexofgoodsamples]
ax.set_xscale('log')
xlabel = ax.set_xlabel('$1-R^2$')
NGOOD = len(indexofgoodsamples)
ax.legend(title=f'{NGOOD} good samples')

In [ ]:
fig, ax = plt.subplots()
B0range = [PBS.B0[PBS.B0>0].min(), PBS.B0.max()]
ax.plot(B0range, B0range)
ax.plot(fit_results_df.B_murn[indexofgoodsamples], PBS.B0[indexofgoodsamples], 'o')
ax.set_xlabel(r'$B_0$ from fit results')
ax.set_ylabel(r'$B_0$ from briefsummaries')

# Differences between fits and available data for bad samples

In [ ]:
( ( fit_results_df.B_murn[indexofbadsamples]==0 ) ).sum()

In [ ]:
theweird = fit_results_df[fit_results_df.B_murn == 0].index

In [ ]:
for params, curve in EVcurves[theweird].iloc[0].items():
    break

In [ ]:
curve

In [ ]:
plt.scatter(curve['evcurve']['V'], curve['evcurve']['E'])
plt.title(theweird)

In [ ]:
indexofbadsamples.difference(theweird)

In [ ]:
PBS.B0[indexofbadsamples.difference( theweird )]

In [ ]:
diff_fit_to_dataset = ((PBS.B0[indexofbadsamples.difference(theweird)] - fit_results_df.B_murn[indexofbadsamples.difference(theweird)])/fit_results_df.B_murn[indexofbadsamples.difference(theweird)]).abs().to_frame().rename(columns={0: 'B0'})

In [ ]:
diff_fit_to_dataset['E0'] = ((PBS.E0[indexofbadsamples] - fit_results_df.E_murn[indexofbadsamples])/fit_results_df.E_murn[indexofbadsamples]).abs().dropna()

In [ ]:
diff_fit_to_dataset['V0'] = ((PBS.V0[indexofbadsamples] - fit_results_df.V_murn[indexofbadsamples])/fit_results_df.V_murn[indexofbadsamples]).abs().dropna()

In [ ]:
large_diff_E0 = diff_fit_to_dataset.query('E0 > 1e-5').index
fig, ax = plt.subplots()
hist, bins = np.histogram(diff_fit_to_dataset.E0, bins=100)
logbins = np.logspace(np.log10(bins[0]), np.log10(bins[-1]), len(bins))
loghist = plt.hist(diff_fit_to_dataset.E0, bins=logbins)
fig = plt.xscale('log')
fig = plt.yscale('log')
for index in large_diff_E0:
    x = diff_fit_to_dataset.E0[index]
    y = 1
    ax.annotate(index, (x, y), rotation='90')
plt.xlabel(r'$|(E_0 ^{fit} - E_0 ^{BS})/E_0^{BS}|$')

In [ ]:
#large_diff_V0 = diff_fit_to_dataset.query('V0 > 0.2e-3').index
fig, ax = plt.subplots()
hist, bins = np.histogram(diff_fit_to_dataset.V0, bins=100)
logbins = np.logspace(np.log10(bins[0]), np.log10(bins[-1]), len(bins))
loghist = ax.hist(diff_fit_to_dataset.V0, bins=logbins)
#for index in large_diff_V0:
#    x = diff_fit_to_dataset.V0[index]
#    y = 1
#    ax.annotate(index, (x, y), rotation=90)
plt.xscale('log')
plt.xlabel(r'$|(V_0 ^{fit} - V_0 ^{BS})/V_0^{BS}|$')

In [ ]:
large_diff_B0 = diff_fit_to_dataset.query('B0 > 0.2e-1').index
fig, ax = plt.subplots()
hist, bins = np.histogram(diff_fit_to_dataset.B0, bins=100)
logbins = np.logspace(np.log10(bins[0]), np.log10(bins[-1]), len(bins))
loghist = plt.hist(diff_fit_to_dataset.B0, bins=logbins)
for index in large_diff_B0:
    x = diff_fit_to_dataset.B0[index]
    y = 1
    ax.annotate(index, (x, y), rotation=90)
plt.xscale('log')
plt.xlabel(r'$|(B_0 ^{fit} - B_0 ^{BS})/B_0^{BS}|$')

In [ ]:
if make_plots:
    figs, axs = plot_curves(EVcurves[large_diff_B0], Fits[large_diff_B0], R2[large_diff_B0])
    for fig, ax  in zip(figs, axs):
        index = ax.title.get_text()
        V0, E0 = PBS[['V0','E0']].loc[index].values
        for paramspec, fitsparam in Fits[index].items():
            break
        e0 = fitsparam[0]
        v0 = fitsparam[-1]
    #    ax = ax.inset_axes([0.45, 0.3, 0.2, 0.2])
        ax.plot([V0], [E0], 'dk', label='value in briefsummary')
        ax.plot([v0],[e0], 'sb', label = 'value from fit')
        ax.legend()

# correct bad samples with new E0, B0, E0

# Test On obvious ones

In [ ]:
Febcc = 'Fe_pv1.bcc.FM'

In [ ]:
Better_evcurves_df.loc[Mo_R].IsGood

In [ ]:
Better_evcurves_df.loc['Fe_pv1.bcc.FM'].IsGood

In [ ]:
plot_curves(EVcurves[[Mo_R, Febcc]], Fits[[Mo_R, Febcc]], R2[[Mo_R, Febcc]])

In [ ]:
GoodBS = PBS.loc[indexofgoodsamples]
GoodBS = GoodBS[~GoodBS.index.str.contains('sigma-BABBB.FM|sigma-AABBB.FM|mu-BBBBA.FM')]
BadBS = PBS.loc[indexofbadsamples]

# Try to correct the bad fits by removing points 

In [ ]:
from importlib.machinery import SourceFileLoader

In [ ]:
len(PBS)

In [ ]:
len(indexofgoodsamples)

In [ ]:
len(indexofbadsamples)

In [ ]:
from importlib.machinery import SourceFileLoader
find_the_good_curve_inside = SourceFileLoader('find_the_good_curve_inside','Tools/DatasetTools/EVCurvesTools.py').load_module().find_the_good_curve_inside
is_common_sense_evcurve = SourceFileLoader('is_common_sense_evcurve','Tools/DatasetTools/EVCurvesTools.py').load_module().is_common_sense_evcurve
plot_the_sample = SourceFileLoader('plot_the_sample','Tools/DatasetTools/EVCurvesTools.py').load_module().plot_the_sample
plot_curves = SourceFileLoader('plot_curves','Tools/DatasetTools/EVCurvesTools.py').load_module().plot_curves
ev_per_angstrom3_to_GPA = 160.21

doexit = False

fixedevcurves = pd.Series([], name='FixedEVcurves')
fixedr2 = pd.Series([], name='FixedR2')
fixedfit = pd.Series([], name='Fixedfit')
tol = 1e-4
now_is_good = []
common_sense_evcurve = []
progress = tqdm(EVcurves[indexofbadsamples].items(), total = len(indexofbadsamples))
for index, paramcurve in progress:
    if index in fixedevcurves.keys():
        continue
    progress.set_description('index')
    if index in GoodBS.index:
        continue
    for paramspec, curvedata in paramcurve.items():
        r2, params_onreduced, reducedv, reducede = find_the_good_curve_inside(curvedata, tol=tol, reset_guess_params = True)
        common_sense_evcurve.append( is_common_sense_evcurve(reducedv, reducede, params_onreduced, unitsofb0='GPa'))
        now_is_good.append( (abs(r2-1) < tol) & common_sense_evcurve[-1] )
        progress.set_postfix_str(f'{index}, 1-r2 = {1-r2:.2e}, B0={params_onreduced[1]}, now is good = {now_is_good[-1]}')
        if params_onreduced[1] < 0 and now_is_good:
            raise ValueError('B0 is negative on '+index)
        fixedevcurves[index] = {
            paramspec: {
                'evcurve' :{ 'V' : reducedv , 'E': reducede },
                'ev_fit_results' :  {'E_murn': params_onreduced[0], 'V_murn' : params_onreduced[-1], 'B_murn': params_onreduced[1], 'Bdev_murn' : params_onreduced[2]},
                'r2' : r2,
                'IsGood' : now_is_good[-1],
                'fit': params_onreduced,
                'calc_param' : paramspec
            }
        }

In [ ]:
sum(now_is_good)

In [ ]:
if make_plots:
    for index, fixedevcurve in fixedevcurves.items():
        if index not in large_diff_B0:
            continue
        #ax[0].plot(reducedv[bestcombi], reducede[bestcombi], 'o')
        for paramspec, data in fixedevcurve.items():
            isgood = data['IsGood']
            x =data['evcurve']['V']
            xx = np.linspace(x.min(), x.max(), 100)
            y = data['evcurve']['E']
            thisparams = copy.copy(data['fit'])
            thisparams[1]/=ev_per_angstrom3_to_GPA
            fixed_r2 = data['r2']
            B0 = thisparams[1]#data['fit'][1]
            V0 = thisparams[-1] # data['fit'][-1]
            message = f'fixed curve, $1-R^2$ = {1-fixed_r2:.1e}, $B_0 $= {B0*ev_per_angstrom3_to_GPA:.3f}, Isgood = {isgood}, v0 = {V0:.3f}'
        if not isgood :
            continue
            fig, ax = plot_curves(EVcurves[[index]], Fits[[index]],R2[[index]])
            l = ax[0].plot(xx, birchmurnaghan(xx, *thisparams), '--',label=message)[0]
            ax[0].plot(x, y, 'o', c=l.get_color())
            ax[0].legend()

In [ ]:
fixedevcurves

## check new total of good curves

In [ ]:
fixedevcurves_df = pd.DataFrame.from_dict(fixedevcurves.to_dict(), orient='index')

In [ ]:
fixedevcurves_df.shape

In [ ]:
Better_fixedevcurves = {}

In [ ]:
for index, evcurves in fixedevcurves_df.iterrows():
    nonan_evcurves = evcurves.dropna()
    if len(nonan_evcurves) > 1:
        break
    nonan_evcurves[nonan_evcurves.index][0]['calc_param'] = nonan_evcurves.index[0]
    Better_fixedevcurves[index] = nonan_evcurves[nonan_evcurves.index][0]

In [ ]:
Better_fixedevcurves_df = pd.DataFrame.from_dict(Better_fixedevcurves, orient='index')

In [ ]:
fixedR2 = pd.Series([])
for index, curve in fixedevcurves.items():
    for paramspec, curvedata in curve.items():
        fixedR2[index] = {paramspec: curvedata['r2']}

In [ ]:
fixedFits = pd.Series([])
for index, curve in fixedevcurves.items():
    for paramspec, curvedata in curve.items():
        fixedFits[index] = {paramspec: curvedata['fit']}

In [ ]:
fixed_ev_fit_results = {} #pd.DataFrame([])
for index, curve in fixedevcurves.items():
    for paramspec, curvedata in curve.items():
        fixed_ev_fit_results[index] = pd.Series(curvedata['ev_fit_results'])
fixed_ev_fit_results_df = pd.DataFrame.from_dict(fixed_ev_fit_results, orient='index')

In [ ]:
indexoffixedgoodsamples = Better_fixedevcurves_df.query('IsGood == True').index

In [ ]:
len(indexoffixedgoodsamples)

In [ ]:
indexoffixedbadsamples = Better_fixedevcurves_df.index.difference(indexoffixedgoodsamples)

In [ ]:
len(indexoffixedbadsamples)

In [ ]:
plot_the_sample = SourceFileLoader('plot_the_sample','Tools/DatasetTools/EVCurvesTools.py').load_module().plot_the_sample
plot_curves = SourceFileLoader('plot_curves','Tools/DatasetTools/EVCurvesTools.py').load_module().plot_curves
fig = plot_curves(fixedevcurves[indexoffixedbadsamples], fixedFits[indexoffixedbadsamples], fixedR2[indexofbadsamples])

In [ ]:
len(indexoffixedgoodsamples)

In [ ]:
PBS.shape

In [ ]:
len(indexoffixedgoodsamples) + len(indexofgoodsamples)#+len(indexoffixedbadsamples)

In [ ]:
indexoffixedgoodsamples[indexoffixedgoodsamples.str.contains('R')]

In [ ]:
finalindexofsamples = indexofgoodsamples.append(indexoffixedgoodsamples)

In [ ]:
finalindexofsamples

In [ ]:
fixedGoodBS = PBS.loc[finalindexofsamples]

In [ ]:
fixedGoodBS.shape

In [ ]:
fixedGoodBS[fixedGoodBS.index.str.contains('R')]

# fixed quantities

In [ ]:
thebins = np.linspace(160, 300, 101)
ax = ev_fit_results_df.B_murn[indexofgoodsamples].plot.hist(bins=thebins)
fixed_ev_fit_results_df.B_murn[indexoffixedgoodsamples].plot.hist(bins=thebins, ax = ax, alpha=0.5, )
fixed_ev_fit_results_df.B_murn[indexoffixedgoodsamples].plot.hist(bins=thebins, ax=ax)
ax.legend([f'{len(indexoffixedgoodsamples)} fixed good samples', f'{len(indexofgoodsamples)} good samples', f'{len(indexoffixedbadsamples)} fixed bad samples'])

# still some bad samples after trying to fix ! 

In [ ]:
len(indexoffixedbadsamples)

In [ ]:
if make_plots:
    for stilbadsample in indexoffixedbadsamples:
        fig, ax=plot_curves(fixedevcurves[[stilbadsample]], fixedFits[[stilbadsample]], fixedR2[[stilbadsample]])
        ax[0].plot(Better_evcurves_df.loc[stilbadsample]['evcurve']['V'],Better_evcurves_df.loc[stilbadsample]['evcurve']['E'] , 'o', markersize=12, markerfacecolor='None')
        handles, labels = ax[0].get_legend_handles_labels()
        thisb0 = fixed_ev_fit_results_df.B_murn[stilbadsample]
        thisbdev = fixed_ev_fit_results_df.Bdev_murn[stilbadsample]
        labels[0] +=f', B0 = {thisb0:.3f}, Bdev = {thisbdev:.3f}'
        ax[0].legend(handles, labels)

# redefine the new good BS

but now I need to correct the E0, B0 and V0 to the results of the fix !

In [ ]:
fixedGoodBS['E0'].dropna()

In [ ]:
fixedGoodBS['E0'][indexoffixedgoodsamples] = fixed_ev_fit_results_df['E_murn'][indexoffixedgoodsamples]

In [ ]:
fixedGoodBS['V0'][indexoffixedgoodsamples] = fixed_ev_fit_results_df['V_murn'][indexoffixedgoodsamples]

In [ ]:
fixedGoodBS['B0'][indexoffixedgoodsamples] = fixed_ev_fit_results_df['B_murn'][indexoffixedgoodsamples]

In [ ]:
fixedGoodBS.dropna().describe()

In [ ]:
fixedGoodBS.shape

## Remove extra magnetic sampling

First feature to remove from this dataset is the list of samples used form ferrimagnetic phase sampling. This subset is not in the current interest and might bring problems so we remove from datastet.

In [ ]:
fixedGoodBS.query('index.str.contains("[UD]+$")')#.loc[['Fe_pv30.sigma.FM', 'Fe_pv30.sigma.NM']]

## Obtain some info from indexes

In [ ]:
Features = Featurizer(fixedGoodBS)

In [ ]:
fixedGoodBS.shape

# TODo this sould be in tools, as a phase cleaner

In [ ]:
from dependencies.bopfoxfeaturizer.BopFoxFeaturizer.struct_db import struct_db
#struct_db = SourceFileLoader('struct_db','BopFoxFeaturizer/struct_db.py').load_module().struct_db
strucdic = struct_db().strucstrings

Target_Class = pd.Series(
    fixedGoodBS.index.str.split('.').map(lambda l: l[1]).map(lambda s: s.split('-')[0]),
    index=fixedGoodBS.index
)
Target_Class[Target_Class.map(lambda s: s in strucdic['list.hcp'])]='hcp'
Target_Class[Target_Class.map(lambda s: s in strucdic['list.fcc'])]='fcc'
Target_Class[Target_Class.map(lambda s: s in strucdic['list.bcc'])]='bcc'
Target_Class[Features.Struc == 'hcp'] = 'hcp'
Target_Class[Features.Struc == 'bcc'] = 'bcc'
Target_Class[Features.Struc == 'fcc'] = 'fcc'
Target_Class[Features.Struc.str.contains('SQS-fcc')] = 'fcc'
Target_Class[Features.Struc.str.contains('SQS-L12')] = 'fcc'
Target_Class[Features.Struc.str.contains('sigma_')] = 'sigma'

Target_Class[    
    Target_Class.str.contains('Al42W') |\
    Target_Class.str.contains('Al9Co2') |\
    Target_Class.str.contains('Al5W') |\
    Target_Class.str.contains('Al12W') |\
    Target_Class.str.contains('Al4W') |\
    Target_Class.str.contains('Al5Co2')
] = 'others'

In [ ]:
DescriptorsLoc = os.path.join(dataset,'Descriptors')
if not os.path.exists(DescriptorsLoc):
    os.makedirs(DescriptorsLoc)
Target_Class.to_pickle(os.path.join(DescriptorsLoc, 'ClassLabels.pkl'))

In [ ]:
fixedGoodBS['Phase'] = Target_Class

In [ ]:
fixedGoodBS.describe()

# force magnetic R out

In [ ]:
fmr = fixedGoodBS.query('Phase == "R" and Mag == "FM"').index

In [ ]:
fixedGoodBS.drop(fmr, inplace=True)

# Save good fixed BS 

In [ ]:
fixedGoodBS.to_pickle(os.path.join(dataset, "CuratedParsedBriefSummary.pkl") )

# some E-V curves, good and bad

## sample bad

In [ ]:
sample_bad_index = BadBS.sample(n=5).index#.append(pd.Index(['Mo_sv53.R.NM'])).unique()

In [ ]:
sample_bad = EVcurves[sample_bad_index] #[BadBS.index].dropna().sample(n=min(5, BadBS.shape[0]))

In [ ]:
plt.rc('font', size=22)
plt.rc('figure', figsize=(12,8))

In [ ]:
sample_bad_r2 = R2[sample_bad.index]

In [ ]:
sample_bad_fit = Fits[sample_bad.index]

In [ ]:
figurecollection, axcollection  = plot_curves(sample_bad, sample_bad_fit, sample_bad_r2)

# Cases of interest

In [ ]:
interesting_cases = ['Fe_pv28Mo_sv1.chi-BAAA.FM', 'Fe_pv11Mo_sv2.mu-AAABA.FM', 'Fe_pv1.fcc.FM', 'Fe_pv1.fcc.FM', Mo_R]

In [ ]:
Better_evcurves_df.IsGood[interesting_cases]

In [ ]:
Fits.loc[interesting_cases][0]

In [ ]:
Better_evcurves_df.ev_fit_results[interesting_cases][0]

In [ ]:
#figs, axs = plot_curves(EVcurves[interesting_cases], Fits[interesting_cases], R2[interesting_cases])


# Sample good

In [ ]:
sample_good = EVcurves[fixedGoodBS.index].dropna().sample(n=5)

In [ ]:
sample_good

In [ ]:
sample_good_r2 = R2[sample_good.index]

In [ ]:
sample_good_fit = Fits[sample_good.index]

In [ ]:
figurecollection, axcollection  = plot_curves(sample_good, sample_good_fit, sample_good_r2)

In [ ]:
sample_max_B0 = fixedGoodBS.query('nelem == 1').B0.idxmax()

In [ ]:
sample_min_B0 = fixedGoodBS.query('nelem == 1').B0.idxmin()

In [ ]:
fixedGoodBS.B0[[sample_min_B0]]

In [ ]:
fixedGoodBS.B0[[sample_max_B0]]

In [ ]:
theminbo = EVcurves_df.loc[sample_min_B0].dropna()

In [ ]:
theminbo[0]

In [ ]:
selection = ((fixedGoodBS.B0>=fixedGoodBS.B0[sample_min_B0]) & (fixedGoodBS.B0<=fixedGoodBS.B0[sample_max_B0]))

In [ ]:
selection.sum()

In [ ]:
selection_stricter = ((fixedGoodBS.B0>1.1*fixedGoodBS.B0[sample_min_B0]) & (fixedGoodBS.B0<0.90*fixedGoodBS.B0[sample_max_B0]) &
                     (list(R2[sample_max_B0].values())[0] > 0.998))

In [ ]:
selection_stricter

In [ ]:
samples_wrong_b0 = fixedGoodBS[~selection].index

In [ ]:
len(samples_wrong_b0)

In [ ]:
fixedGoodBS.B0.plot.hist(bins=100)

In [ ]:
curve_wrong_b0 = EVcurves[samples_wrong_b0]

In [ ]:
r2_wrong_b0 = R2[samples_wrong_b0]

In [ ]:
fits_wrong_b0 = Fits[samples_wrong_b0]

In [ ]:
curve_wrong_b0

In [ ]:
figurecollection, axcollection  = plot_curves(curve_wrong_b0, fits_wrong_b0, r2_wrong_b0)
for ax, index  in zip(axcollection, samples_wrong_b0):
    title=ax.get_title()
    title += f'$B_0$ = {fixedGoodBS.B0[index]}'
    ax.set_title(title)

In [ ]:
GoodBS.drop(index=samples_wrong_b0, inplace=True)

In [ ]:
samples_low_b0 = GoodBS.query('B0 < 150').index

In [ ]:
curve_low_b0 = EVcurves[samples_low_b0]

In [ ]:
r2_low_b0 = R2[samples_low_b0]

In [ ]:
fits_low_b0 = Fits[samples_low_b0]

In [ ]:
figurecollection, axcollection  = plot_curves(curve_low_b0, fits_low_b0, r2_low_b0)
for ax, index  in zip(axcollection, samples_low_b0):
    title=ax.get_title()
    title += f'$B_0$ = {GoodBS.B0[index]}'
    ax.set_title(title)

# Save for later use 

In [ ]:
#curatedbs = os.path.join(dataset,'CuratedParsedBriefSummary.pkl')
#GoodBS.to_pickle(curatedbs)

In [ ]:
fixedGoodBS.B0.hist(bins=100)

# a bit of a different formulation

In [ ]:
PBS['config'] = PBS.index.str.split('.').map(lambda a: a[1].split('-')[-1])

In [ ]:
PBS['system'] = PBS.index.map(lambda a: '-'.join(re.findall('Fe|Mo', a)))

In [ ]:
PBS.index = pd.MultiIndex.from_frame(PBS[['system', 'Phase', 'config', 'Mag']])

In [ ]:
PBS.set_index(['system', 'Phase', 'config', 'Mag'], inplace=True)

In [ ]:
PBS